# 📈 FolioTrack — Quickstart

This notebook demonstrates the core workflow of the `foliotrack` library:
1. Create a portfolio and buy securities
2. Update live prices from the market
3. Set target allocations
4. Run the MIQP optimizer to find the best trades
5. Backtest the portfolio over historical data
6. Save and load the portfolio

## 1. Setup

In [ ]:
import logging
from foliotrack.domain.Portfolio import Portfolio
from foliotrack.services.MarketService import MarketService
from foliotrack.services.OptimizationService import OptimizationService
from foliotrack.services.BacktestService import BacktestService
from foliotrack.storage.PortfolioRepository import PortfolioRepository

logging.basicConfig(level=logging.INFO)

# Instantiate services
repo = PortfolioRepository()
optimizer = OptimizationService()
backtester = BacktestService()
market_service = MarketService()

## 2. Create Portfolio & Buy Securities

We use affordable EUR-denominated ETFs that are PEA-eligible and give the optimizer room to work with small investment amounts.

In [ ]:
portfolio = Portfolio("Example Portfolio", currency="EUR")

# Buy ETFs — all affordable (€15-45 range)
portfolio.buy_security("LEM.PA", volume=10.0, price=17.0, date="2024-01-15", fill=True)   # Amundi Emerging Markets ~€17
portfolio.buy_security("CW8.PA", volume=10.0, price=30.0, date="2024-01-15", fill=True)   # Amundi MSCI World ~€30
portfolio.buy_security("PANX.PA", volume=5.0, price=45.0, date="2024-03-01", fill=True)   # Amundi Nasdaq-100 ~€45

## 3. Update Prices from Market

In [ ]:
market_service.update_prices(portfolio)

# Display current portfolio state
for info in portfolio.get_portfolio_info():
    print(f"{info['name']:50s}  {info['ticker']:10s}  Price: {info['price_in_security_currency']:>8.2f}{info['symbol']}  Volume: {info['volume']:.0f}")

## 4. Set Target Allocations & Optimize

Set target weights and let the MIQP solver find the optimal integer number of shares to buy.

In [ ]:
# Target allocation: 50% World, 30% Emerging, 20% Nasdaq
portfolio.set_target_share("CW8.PA", 0.5)
portfolio.set_target_share("LEM.PA", 0.3)
portfolio.set_target_share("PANX.PA", 0.2)

In [ ]:
# Optimize: invest €500 with at least 99% utilization
security_counts, total_to_invest, final_shares = optimizer.solve_equilibrium(
    portfolio,
    investment_amount=500.0,
    min_percent_to_invest=0.99,
)

print(f"\nTotal to invest: {total_to_invest:.2f}€")
print("\nRecommended trades:")
for info in portfolio.get_portfolio_info():
    print(f"  {info['ticker']:10s}  Buy: {info['volume_to_buy']:>3} units  ({info['amount_to_invest']:>8.2f}€)  Final share: {info['final_share']:.2%}")

## 5. Backtest

In [ ]:
result = backtester.run_backtest(
    portfolio, market_service, start_date="2020-01-01", end_date="2025-01-01"
)
result.display()

## 6. Save & Load Portfolio

In [ ]:
# Save
repo.save_to_json(portfolio, "../Portfolios/investment_example.json")

# Load it back
loaded = repo.load_from_json("../Portfolios/investment_example.json")
print(f"Loaded portfolio: {loaded.name}, {len(loaded.securities)} securities")

## 7. Interactive Dashboard

For a full visual experience, launch the Streamlit dashboard:

```bash
pip install foliotrack[dashboard]
dash
```